# 🔌 Class 7: Agents in Action: Tools, MCP & Guardrails
### One standard plug, and how we keep agents safe · Day 4 · Morning · 2 hours

**Same story.** Yesterday Neo learned to use three tools that *we* wired in by hand. A real company has dozens of systems, and every new one would need its own custom connection.

**Today's question:** *If every new system needs its own custom connection, it doesn't scale. How does the industry connect agents to systems in a standard way, and how do we keep them safe?*

| Part | Topic | Cells |
|---|---|---|
| A | The problem and the plug (MCP) | S, 1 |
| B | The 3 things a server offers | 2–4 |
| C | The agent over MCP | 5–7 |
| D | Guardrails | 8–10 |
| E | Your turn | 11–12 |

> 💡 **Key terms come first.** Each part starts with a key-terms slide. Then you run the cells.
> 🔒 All data is **mock** (fictional company *Nova Industries*). The "SAP" here is two CSV files behind a small **MCP server**: a Python file the trainer prepared (`mcp_server/sap_orders_server.py`).

## ⚙️ ▶ CELL S: Setup
Checks that Ollama is running, then **starts the mock SAP Orders MCP server** and shakes hands with it. Run once.

In [1]:
# ▶ CELL S · SETUP: check Ollama, then start the mock SAP Orders MCP server. Run once.
import sys, os, re, json
sys.path.insert(0, "../../shared")            # where bootcamp_llm.py and mcp_client.py live
from bootcamp_llm import *
from mcp_client import *

SERVER = [sys.executable, "../mcp_server/sap_orders_server.py"]      # the MCP server: a Python file the trainer prepared

def connect(*flags):
    """Start the MCP server as a child process and shake hands with it."""
    return McpClient(SERVER + list(flags))

check_setup()
print("🤖 Neo says:", ask_ai("Reply with exactly these words: Connection OK", temperature=0, max_tokens=10))
client = connect()                            # read-only by default
print(f"🔌 Connected to MCP server: {client.server_info['title']} (v{client.server_info['version']})")

✅ Ollama 0.40.13 is running at http://localhost:11434
✅ Main model : qwen3:8b
✅ Small model: llama3.2:1b
⚙️  Config: c:\Users\rajuk\OneDrive\Desktop\Day_4\Day_4\bootcamp_config.json
🤖 Neo says: Connection OK
🔌 Connected to MCP server: Nova SAP Orders (mock) (v1.0.0)


---
# A · The problem and the plug
**In real life:** SAP does not have a custom cable to every other system. It uses **standard interfaces** (IDoc, OData, RFC), so a new system plugs into a known shape.

Agents had the same problem. With 4 agents and 5 systems, custom connections mean **4 × 5 = 20** pieces to build and maintain. With one standard plug it is **4 + 5 = 9**.

> 🔌 **MCP (Model Context Protocol) = "USB-C for AI".** One standard plug, so any AI agent can connect to any system that speaks it.

### ▶ CELL 1: Connect, and ask the server what it offers

In [2]:
# ▶ CELL 1 · CONNECT: the handshake, then DISCOVER what the server offers (nobody typed this list)
print("Server   :", client.server_info["title"], "· version", client.server_info["version"])
print("Protocol :", client.protocol)
print("Offers   :", list(client.capabilities))

tools = client.list_tools()
print(f"\n{len(tools)} tools discovered:")
for t in tools:
    print(" •", t["name"])

Server   : Nova SAP Orders (mock) · version 1.0.0
Protocol : 2025-06-18
Offers   : ['tools', 'resources', 'prompts']

6 tools discovered:
 • list_orders
 • get_order
 • get_vendor
 • check_approval
 • delayed_summary
 • create_ticket


> 🤝 **initialize** is the handshake: client and server agree on the protocol and on what each side can do. Then **tools/list** is *discovery*. Yesterday we typed the tool menu by hand (Class 6, Cell 3). Today **the server tells us**.

---
# B · The 3 things a server offers
**Tools** (actions the AI may request) · **Resources** (data to read) · **Prompts** (ready-made requests).

### ▶ CELL 2: Tools: what they do, and whether they only read

In [3]:
# ▶ CELL 2 · TOOLS: each one says what it needs and whether it only READS
for t in tools:
    label = "🔍 read-only" if t["annotations"]["readOnlyHint"] else "✏️  changes data"
    print(f"{label:<15} {t['name']}({', '.join(t['inputSchema']['properties'])})")

print("\nCalling a tool by hand:")
print(client.call_tool("get_order", {"po_number": "4500012345"}))
print("\nA tool problem comes back as a readable message, not a crash:")
print(client.call_tool("get_order", {"po_number": "4500099999"}))

🔍 read-only     list_orders(plant, status, vendor_id)
🔍 read-only     get_order(po_number)
🔍 read-only     get_vendor(vendor_id)
🔍 read-only     check_approval(value_inr)
🔍 read-only     delayed_summary()
✏️  changes data create_ticket(po_number, note)

Calling a tool by hand:
{'ok': True, 'data': {'po_number': '4500012345', 'vendor_id': 'V100', 'vendor_name': 'Alpha Supplies', 'plant': '1000', 'material': 'M-100 Steel Sheet', 'qty': 100, 'value_inr': 50000, 'delivery_date': '2026-10-01', 'status': 'Blocked', 'block_reason': 'Credit check'}}

A tool problem comes back as a readable message, not a crash:
{'ok': False, 'data': 'PO 4500099999 not found.'}


> 🏷️ Every tool carries a **read-only label**. That label is what lets a program decide *when to ask a person* (Part D). Notice the one tool that **changes data**.

### ▶ CELL 3: Resources and prompts

In [4]:
# ▶ CELL 3 · RESOURCES (data to read) and PROMPTS (ready-made requests)
print("RESOURCES:")
for r in client.list_resources():
    print(f" • {r['uri']:<26} {r['name']}")
print("\norders://summary →", client.read_resource("orders://summary"))

print("\nPROMPTS:")
for p in client.list_prompts():
    print(f" • {p['name']:<22} {p['description']}")
print("\nThe 'blocked_orders_report' prompt for plant 1000 turns into this request:\n")
print(client.get_prompt("blocked_orders_report", {"plant": "1000"}))

RESOURCES:
 • orders://all               All purchase orders (CSV)
 • orders://summary           Order summary
 • policy://approval-limits   PO approval limits

orders://summary → {"orders": 12, "by_status": {"Blocked": 4, "Delayed": 4, "Open": 2, "Delivered": 2}, "by_plant": {"1000": {"orders": 7, "value_inr": 3420000}, "2000": {"orders": 5, "value_inr": 1295000}}}

PROMPTS:
 • blocked_orders_report  Report of blocked purchase orders for one plant, with the reason for each block.
 • vendor_followup        Draft (do not send) a polite follow-up message to the vendor of a purchase order.

The 'blocked_orders_report' prompt for plant 1000 turns into this request:

Use the nova-sap-orders tools to list the BLOCKED purchase orders for plant 1000. Show a table with PO number, vendor, value in INR and block reason. Use only facts from the tools. Do not change anything.


> 📚 A **resource** is something to *read* (like a report). A **tool** is something to *do* (like a BAPI). A **prompt** is a *saved request* the server offers.

### ▶ CELL 4: What one tool call looks like on the wire

In [5]:
# ▶ CELL 4 · ON THE WIRE: one tool call is two plain JSON messages (the "id" number will differ on your screen)
client.wire.clear()
client.call_tool("check_approval", {"value_inr": 1200000})
for direction, message in client.wire:
    print(direction, json.dumps(message, ensure_ascii=False), "\n")

→ {"jsonrpc": "2.0", "id": 9, "method": "tools/call", "params": {"name": "check_approval", "arguments": {"value_inr": 1200000}}} 

← {"jsonrpc": "2.0", "id": 9, "result": {"content": [{"type": "text", "text": "{\"value_inr\": 1200000, \"approver\": \"Director\", \"policy\": \"3.3\"}"}], "isError": false}} 



> 🔎 **→** is the request, **←** is the result. No magic: JSON in, JSON out. Any agent that speaks this can use any server that speaks it. *That is the whole idea.*

---
# C · The agent over MCP
Same agent as Class 6, with two changes: the **menu comes from the server**, and a **gate** sits between "the AI asked" and "the program runs it".

### ▶ CELL 5: The tool menu, built from discovery

In [6]:
# ▶ CELL 5 · THE MENU: built from what the server told us, plus the instructions
ROLE = "You are Neo, an SAP procurement assistant for Nova Industries."
RULES = ["Answer using ONLY facts returned by your tools. Never guess a status, a contact, a value or an approver.",
         "If a tool returns an error, read it and try again, or say what is missing."]
FORMAT = ""               # optional: how the final answer should look (the Class 8 capstone sets this)

ALLOWED = None            # None = every tool the server offers. Or a set of names: an allow-list.
MAX_STEPS = 6             # the step limit: an agent must never loop forever

def offered():
    """The tools the server offers RIGHT NOW (name -> description, arguments, read-only label). Nobody types this list."""
    return {t["name"]: t for t in client.list_tools() if ALLOWED is None or t["name"] in ALLOWED}

def system_prompt():
    menu = "\n".join(f"- {n}({', '.join(t['inputSchema']['properties'])}): {t['description']}" for n, t in offered().items())
    rules = "\n".join(f"- {r}" for r in RULES)
    return f"""{ROLE}
RULES:
{rules}

TOOLS (the only ones you may use):
{menu}

Reply with ONE JSON object and nothing else.
To use a tool:  {{"thought": "why I need it", "tool": "<tool name>", "args": {{"<argument>": "<value>"}}}}
To finish:      {{"thought": "why I can finish", "answer": "<final answer for the user>"}}
{FORMAT}""".strip()

print(system_prompt())

You are Neo, an SAP procurement assistant for Nova Industries.
RULES:
- Answer using ONLY facts returned by your tools. Never guess a status, a contact, a value or an approver.
- If a tool returns an error, read it and try again, or say what is missing.

TOOLS (the only ones you may use):
- list_orders(plant, status, vendor_id): List purchase orders. Optional filters: plant ('1000' or '2000'), status (Open, Delayed, Blocked, Delivered), vendor_id (V100 to V400). Each order has po_number, vendor, plant, material, value_inr, delivery_date, status and block_reason.
- get_order(po_number): Full details of ONE purchase order: status, block reason, value (INR), delivery date, plant, vendor id.
- get_vendor(vendor_id): Vendor name, contact person, email and rating for a vendor_id (for example V200). Get the vendor_id from get_order or list_orders.
- check_approval(value_inr): Who must approve a purchase order of this value in INR (Procurement Policy section 3).
- delayed_summary(): Number and

> 📋 Compare with Class 6, Cell 3: there we typed `TOOLS = {...}`. Now `offered()` **asks the server**. Add a tool to the server and it appears on this menu by itself.

### ▶ CELL 6: The loop: think → act → observe, over MCP

In [7]:
# ▶ CELL 6 · THE AGENT LOOP over MCP: think → act → observe → repeat
def gate(tool, args):
    """GUARDRAIL between 'the AI asked' and 'the program runs it'. First version: read-only tools only."""
    return bool(tool["annotations"].get("readOnlyHint"))

def execute(reply):
    """The PROGRAM's side: check the request, then ask the MCP server to run it."""
    name, args = reply.get("tool"), reply.get("args")
    tools = offered()
    if name not in tools:
        return {"error": f"Unknown tool '{name}'. Allowed tools: {list(tools)}"}
    if not isinstance(args, dict):
        return {"error": 'args must be an object, for example {"po_number": "4500012345"}'}
    if not gate(tools[name], args):
        return {"error": f"Blocked by a guardrail: '{name}' is not allowed to run."}
    result = client.call_tool(name, args)
    return result["data"] if result["ok"] else {"error": result["data"]}

def run_agent(question, max_steps=None, verbose=True):
    max_steps = max_steps or MAX_STEPS
    messages = [{"role": "system", "content": system_prompt()}, {"role": "user", "content": question}]
    used, evidence = [], []
    for step in range(1, max_steps + 1):
        try:
            reply = chat_json(messages)                                  # THINK
        except ValueError as e:                                          # the model broke the JSON format
            if verbose: print(f"⚠️  step {step}: {e}")
            messages.append({"role": "user", "content": "Reply with ONE valid JSON object only."})
            continue
        if not isinstance(reply, dict):
            messages.append({"role": "user", "content": "Reply with ONE JSON object only."})
            continue
        if verbose: print(f"🧠 THINK   (step {step}): {reply.get('thought', '')}")
        if "answer" in reply:
            if verbose: print(f"✅ ANSWER : {reply['answer']}")
            return {"answer": str(reply["answer"]), "tools": used, "steps": step, "stopped": False, "evidence": evidence, "approval_tools": needs_approval(used)}
        observation = execute(reply)                                     # ACT
        used.append(reply.get("tool"))
        evidence.append({"tool": reply.get("tool"), "args": reply.get("args"), "result": observation})
        if verbose:
            print(f"🛠️  ACT     : {reply.get('tool')}({reply.get('args')})")
            print(f"👀 OBSERVE: {json.dumps(observation, ensure_ascii=False)[:300]}")
        messages += [{"role": "assistant", "content": json.dumps(reply, ensure_ascii=False)},          # OBSERVE
                     {"role": "user", "content": "Tool result: " + json.dumps(observation, ensure_ascii=False)}]
    if verbose: print("⛔ Stopped: step limit reached")                   # a guardrail
    return {"answer": None, "tools": used, "steps": max_steps, "stopped": True, "evidence": evidence, "approval_tools": needs_approval(used)}

def needs_approval(tool_names):
    """Which of these tools change data (so a person must approve them)? The server labels every tool."""
    tools = {t["name"]: t for t in client.list_tools()}
    return [n for n in tool_names if n in tools and not tools[n]["annotations"].get("readOnlyHint", False)]

QUESTION = "Which orders for Plant 1000 are blocked and why?"      # ← change me
_ = run_agent(QUESTION)

🧠 THINK   (step 1): I need to list blocked orders for Plant 1000 to find out which ones are blocked and the reasons.
🛠️  ACT     : list_orders({'plant': '1000', 'status': 'Blocked'})
👀 OBSERVE: {"count": 4, "orders": [{"po_number": "4500012345", "vendor_id": "V100", "vendor_name": "Alpha Supplies", "plant": "1000", "material": "M-100 Steel Sheet", "qty": 100, "value_inr": 50000, "delivery_date": "2026-10-01", "status": "Blocked", "block_reason": "Credit check"}, {"po_number": "4500012348",
🧠 THINK   (step 2): I have the list of blocked orders for Plant 1000 along with their block reasons. I can now present this information to the user.
✅ ANSWER : {'blocked_orders': [{'po_number': '4500012345', 'vendor': 'Alpha Supplies', 'plant': '1000', 'material': 'M-100 Steel Sheet', 'value_inr': 50000, 'block_reason': 'Credit check'}, {'po_number': '4500012348', 'vendor': 'Chennai Components', 'plant': '1000', 'material': 'M-400 Gearbox', 'value_inr': 1200000, 'block_reason': 'Price variance'}, {'p

> 🔁 The loop is the one from Class 6. Look for the **tool the agent chose** and the **facts in the OBSERVE line**. Expected: four orders on plant 1000 (`…345` credit check, `…348` price variance, `…352` missing goods receipt, `…355` credit check). The wording of the THINK lines will differ on your model.

### ▶ CELL 7: Plug and play: the server gets a new tool

In [8]:
# ▶ CELL 7 · PLUG AND PLAY: the trainer adds a tool to the SERVER. We change NO agent code.
print("Tools before:", len(client.list_tools()))
client.close()
client = connect("--with-scorecard")                  # the same server file, now offering one more tool
print("Tools after :", len(client.list_tools()), "→ new:", [t["name"] for t in client.list_tools() if t["name"] == "vendor_scorecard"])

_ = run_agent("Which vendor has the highest total order value, and how many of its orders are delayed or blocked?")

Tools before: 6
Tools after : 7 → new: ['vendor_scorecard']
🧠 THINK   (step 1): I need to use the vendor_scorecard tool to get the total order value and the number of delayed or blocked orders for each vendor.
🛠️  ACT     : vendor_scorecard({})
👀 OBSERVE: {"Alpha Supplies": {"rating": "A", "orders": 3, "delayed": 0, "blocked": 2, "total_value_inr": 260000}, "Bharat Logistics": {"rating": "B", "orders": 3, "delayed": 2, "blocked": 1, "total_value_inr": 1295000}, "Chennai Components": {"rating": "B", "orders": 3, "delayed": 2, "blocked": 1, "total_valu
🧠 THINK   (step 2): I can finish because the tool result shows that Chennai Components has the highest total order value of 2,700,000 INR, and it has 2 delayed or blocked orders.
✅ ANSWER : Chennai Components has the highest total order value of 2,700,000 INR, and it has 2 delayed or blocked orders.


> ✅ Expected: **Chennai Components**, ₹27,00,000 in total, 2 delayed and 1 blocked. Yesterday (Class 6, Cell 10) we edited the notebook to add a fourth tool. **Today we edited nothing here.** That is plug and play.

---
# D · Guardrails
An agent that can act needs limits. Four to know:
1. **Read-only first**: start with tools that cannot change anything.
2. **Allow-list**: the agent only gets the tools it needs.
3. **Step limit**: no endless loops.
4. **Human approval** before any change, like an SAP **release strategy**.

### ▶ CELL 8: Guardrails 1–3 in action

In [9]:
# ▶ CELL 8 · GUARDRAILS 1-3: read-only first · allow-list · step limit
print("1) READ-ONLY FIRST: the agent asks for a change, the gate says no")
print("  ", execute({"tool": "create_ticket", "args": {"po_number": "4500012345", "note": "Chase the credit check"}}))

print("\n2) ALLOW-LIST: this agent gets only two tools")
ALLOWED = {"get_order", "check_approval"}
try:
    print("   menu :", list(offered()))
    print("  ", execute({"tool": "list_orders", "args": {}}))
finally:
    ALLOWED = None                                   # back to the full menu

print("\n3) STEP LIMIT: a limit of 1 for a question that needs two steps")
_ = run_agent("Who must approve PO 4500012348?", max_steps=1)

1) READ-ONLY FIRST: the agent asks for a change, the gate says no
   {'error': "Blocked by a guardrail: 'create_ticket' is not allowed to run."}

2) ALLOW-LIST: this agent gets only two tools
   menu : ['get_order', 'check_approval']
   {'error': "Unknown tool 'list_orders'. Allowed tools: ['get_order', 'check_approval']"}

3) STEP LIMIT: a limit of 1 for a question that needs two steps
🧠 THINK   (step 1): I need to check the value of the purchase order to determine the approver.
🛠️  ACT     : get_order({'po_number': '4500012348'})
👀 OBSERVE: {"po_number": "4500012348", "vendor_id": "V300", "vendor_name": "Chennai Components", "plant": "1000", "material": "M-400 Gearbox", "qty": 10, "value_inr": 1200000, "delivery_date": "2026-09-30", "status": "Blocked", "block_reason": "Price variance"}
⛔ Stopped: step limit reached


> 🛡️ The **program**, not the AI, decides what runs. Each guardrail is a few lines of ordinary code *between* the AI and the system.

### ▶ CELL 9: Guardrail 4: human approval
Now we let the server write (`--allow-write`), but **only a person can say yes**. Run the cell and answer the prompt. Then run it again and answer **no**.

In [10]:
# ▶ CELL 9 · GUARDRAIL 4: HUMAN APPROVAL: the agent pauses until a person says yes
client.close()
client = connect("--allow-write", "--with-scorecard")   # the server CAN create tickets now, but only after a person approves

def ask_person(name, args):
    """Pause the agent until a human decides. (Set BOOTCAMP_AUTO_APPROVE=yes|no only for automated tests.)"""
    print(f"\n🛑 APPROVAL NEEDED: the agent wants to run  {name}({args})")
    answer = os.environ.get("BOOTCAMP_AUTO_APPROVE") or input("   Approve? Type yes or no: ")
    ok = answer.strip().lower() in ("y", "yes")
    print("   ✅ Approved by a person." if ok else "   ⛔ Declined. Nothing was changed.")
    return ok

def gate(tool, args):
    if tool["annotations"].get("readOnlyHint"):
        return True                                       # reading is safe: let it run
    return ask_person(tool["name"], args)                 # changing data: a person decides

r = run_agent("Create a ticket for PO 4500012345 so someone chases the credit check.")
print("\nTools that needed approval:", r["approval_tools"])

🧠 THINK   (step 1): I need to create a follow-up ticket for the specified purchase order to initiate the credit check process.

🛑 APPROVAL NEEDED: the agent wants to run  create_ticket({'po_number': '4500012345', 'note': 'Credit check required.'})
   ✅ Approved by a person.
🛠️  ACT     : create_ticket({'po_number': '4500012345', 'note': 'Credit check required.'})
👀 OBSERVE: {"ticket_id": "TCK-0008", "po_number": "4500012345", "note": "Credit check required.", "status": "Open", "created": "2026-10-29T23:02:17"}
🧠 THINK   (step 2): The ticket has been successfully created for the purchase order, and it is now open for follow-up.
✅ ANSWER : Ticket TCK-0008 has been created for PO 4500012345 with the note 'Credit check required.' The ticket is now open.

Tools that needed approval: ['create_ticket']


> 🛑 The agent **paused** and waited. The `gate()` function is the only thing we changed; the loop is untouched. If you typed **no**, nothing was created. In SAP terms: a **release strategy** for AI.

### ▶ CELL 10: When NOT to use an agent

In [11]:
# ▶ CELL 10 · AGENT OR PLAIN AUTOMATION? The same job, two ways
reset_usage()
plain = client.call_tool("check_approval", {"value_inr": 85000})["data"]
print("Plain automation:", plain, f"→ model calls: {usage_total['calls']}")

reset_usage()
r = run_agent("Who must approve a purchase order of INR 85000?", verbose=False)
print("Agent           :", r["answer"], f"→ model calls: {usage_total['calls']}, steps: {r['steps']}, {usage_total['seconds']} s")

Plain automation: {'value_inr': 85000, 'approver': 'Line Manager', 'policy': '3.1'} → model calls: 0
Agent           : The Line Manager must approve a purchase order of INR 85000. → model calls: 2, steps: 2, 19.45 s


> ⚖️ Same answer. The fixed rule needed **no model at all**. Use an agent when *the next step depends on what is found*. For a fixed, rule-based process, normal automation is faster, cheaper and easier to audit.

---
# E · Your turn (40 min)
Ask **five business questions** through the MCP-connected agent. For each one, note **which tool it chose** and **whether it needed approval**.

### ▶ CELL 11: Five business questions

In [12]:
# ▶ CELL 11 · Your turn: five business questions through the MCP-connected agent
MY_QUESTIONS = [   # ← change me: replace any of these
    "Which orders for Plant 1000 are blocked and why?",
    "Who is the vendor contact for PO 4500012346?",
    "Who must approve PO 4500012348?",
    "Which vendor has the most delayed order value?",
    "Create a ticket for PO 4500012345 about the credit check.",
]
print(f"{'#':<3}{'Tools the agent chose':<44}{'Steps':<7}Approval needed?")
for i, q in enumerate(MY_QUESTIONS, 1):
    r = run_agent(q, verbose=False)
    print(f"{i:<3}{' → '.join(r['tools']) or '—':<44}{r['steps']:<7}{'YES: ' + ', '.join(r['approval_tools']) if r['approval_tools'] else 'no'}")
    print(f"   {q}\n   ↳ {(r['answer'] or 'no answer: step limit reached')[:150]}\n")

#  Tools the agent chose                       Steps  Approval needed?
1  list_orders                                 2      no
   Which orders for Plant 1000 are blocked and why?
   ↳ {'blocked_orders': [{'po_number': '4500012345', 'vendor_id': 'V100', 'vendor_name': 'Alpha Supplies', 'plant': '1000', 'material': 'M-100 Steel Sheet'

2  get_order → get_vendor                      3      no
   Who is the vendor contact for PO 4500012346?
   ↳ The vendor contact for PO 4500012346 is Rahul Verma.

3  get_order → check_approval                  3      no
   Who must approve PO 4500012348?
   ↳ The approver for PO 4500012348 is the Director.

4  delayed_summary                             2      no
   Which vendor has the most delayed order value?
   ↳ Chennai Components has the most delayed order value with a total of 1,500,000 INR.


🛑 APPROVAL NEEDED: the agent wants to run  create_ticket({'po_number': '4500012345', 'note': 'Credit check required.'})
   ✅ Approved by a person.
5  create

> 💡 Try your own: *Which orders for Plant 2000 are delayed?* · *What does the approval policy say?* (use the resource) · *Delete PO 4500012345.* (what happens?)

### ▶ CELL 12: Scoreboard: check it in code
Question 4 must produce a real ticket id (`TCK-…`). If you type **no** at the approval prompt, nothing is created and that row shows ❌: the guardrail worked, the test just wanted a ticket.

In [13]:
# ▶ CELL 12 · Final check: four questions scored in code (tools used · a word in the answer · approval)
FINAL = [  # (question, tools that must be used, a word the answer must contain, should a person be asked?)
    ("Which orders for Plant 1000 are blocked and why?",           {"list_orders"},                "credit check", False),
    ("Who must approve PO 4500012348?",                            {"get_order", "check_approval"}, "director",    False),
    ("Which vendor has the highest total order value?",            {"vendor_scorecard"},           "chennai",      False),
    ("Create a ticket for PO 4500012345 about the credit check.",  {"create_ticket"},              "tck-",         True),   # a real ticket id: a declined request must NOT score
]
points = 0
for question, need, word, approval in FINAL:
    r = run_agent(question, verbose=False)
    ok = need <= set(r["tools"]) and bool(r["answer"]) and word in r["answer"].lower() and bool(r["approval_tools"]) == approval
    points += ok
    print(f"{'✅' if ok else '❌'} {question}   tools: {r['tools']}   approval: {bool(r['approval_tools'])}")
print(f"\nScore: {points} / {len(FINAL)}")
client.close()

✅ Which orders for Plant 1000 are blocked and why?   tools: ['list_orders']   approval: False
✅ Who must approve PO 4500012348?   tools: ['get_order', 'check_approval']   approval: False
✅ Which vendor has the highest total order value?   tools: ['vendor_scorecard']   approval: False

🛑 APPROVAL NEEDED: the agent wants to run  create_ticket({'po_number': '4500012345', 'note': 'Credit check required.'})
   ✅ Approved by a person.
✅ Create a ticket for PO 4500012345 about the credit check.   tools: ['create_ticket']   approval: True

Score: 4 / 4


---
## ✅ Remember three things
1. **MCP = USB-C for AI.** A server offers **tools, resources and prompts** in one standard shape, so any agent can use any system.
2. **Guardrails keep agents safe:** step limits · read-only first · human approval before any change.
3. **Not every job needs an agent.** Fixed, rule-based processes are better left to normal automation.

> 🌉 **Bridge to Class 8:** We have seen every piece: prompts, knowledge, agents, tools and guardrails. *How do they come together into one real solution?*

---
## 🛠️ Trainer troubleshooting
| Message | Cause | Fix |
|---|---|---|
| `Cannot reach Ollama at http://localhost:11434` | Ollama isn't running | Windows: Start menu → **Ollama**. Mac: open the Ollama app. Or in a terminal: `ollama serve` |
| `Model '…' is not downloaded` | The model in `bootcamp_config.json` isn't pulled | `ollama pull <name>` (see `ollama list`) |
| `ModuleNotFoundError: bootcamp_llm` or `mcp_client` | The notebook is not running from its own folder | Open the **Day_4** folder as the workspace, then pick the **`.venv`** kernel |
| `The MCP server stopped before answering` | The server file crashed or the path is wrong | Run it by hand to see the error: `python Class_7_MCP_Guardrails/mcp_server/sap_orders_server.py` (press Ctrl+C to stop) |
| `Blocked by a guardrail: 'create_ticket' …` | The read-only gate did its job | Expected in CELL 8. CELL 9 swaps in human approval |
| `This server is READ-ONLY` | The server was started without `--allow-write` | Expected until CELL 9, which restarts it with writes enabled |
| `The model did not return valid JSON` | Small models sometimes break the format | Run the cell again. This is a real lesson: agents need validation |
| The agent picks the wrong tool | Normal with small models | Try a larger `model` in the config (7B or 8B follows tool menus better) |
| Antigravity / VS Code do not list the server | The config has no absolute path for this laptop | Run `python setup/make_mcp_config.py`, then restart the editor |